# Diabetes Dataset — Exploratory Data Analysis

This notebook explores the **raw** Pima Indians Diabetes dataset (`diabetes_v1_raw.csv`), the
input to the project's `data_versioning` pipeline. EDA is the first analytical step: it surfaces
data-quality problems and statistical structure, and it **motivates** the cleaning and feature
engineering decisions made downstream.

We deliberately analyse the *raw* version because it still contains physiologically-impossible
**zeros** (e.g. `BMI = 0`, `BloodPressure = 0`) that are really *missing values*. Seeing them is
the whole point.

**The 10 tasks:** overview · summary stats · missingness · class balance · distributions ·
outliers · correlation · feature-vs-target · pairwise relationships · insights.

## Setup

Imports, plotting theme, and data loading. The loader locates the raw CSV produced by
`data_versioning` and falls back to running `build_all.py` if it is missing. Zeros in the medical
columns are converted to `NaN` for analysis (matching the `ZERO_AS_MISSING` list used by the
pipeline's `clean.py`).

In [ ]:
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", None)

# Columns where a 0 is physiologically impossible -> treat as missing.
# Mirrors ZERO_AS_MISSING in data_versioning/scripts/clean.py.
ZERO_AS_MISSING = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

# Resolve repo root from this notebook's location: eda/notebooks/ -> repo root.
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "data_versioning").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

RAW_CSV = REPO_ROOT / "data_versioning" / "outputs" / "diabetes_v1_raw.csv"

if not RAW_CSV.exists():
    print("Raw file missing — regenerating via data_versioning/scripts/build_all.py ...")
    subprocess.run(
        [sys.executable, str(REPO_ROOT / "data_versioning" / "scripts" / "build_all.py")],
        check=True,
    )

df = pd.read_csv(RAW_CSV)
print(f"Loaded {RAW_CSV.relative_to(REPO_ROOT)} -> {df.shape[0]} rows x {df.shape[1]} cols")
df.head()

## 1. Dataset overview

*Why it matters:* before any analysis we confirm the size, the data types, and that the columns
are what we expect — 8 numeric clinical features plus the binary `Outcome` target (1 = diabetes).

In [ ]:
print(f"Shape: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024:.1f} KB\n")
df.info()

**Takeaway:** 768 rows × 9 columns, all numeric, no `NaN`s reported by pandas — but that is
misleading, because the missing values are encoded as zeros (next sections).

## 2. Summary statistics

*Why it matters:* `describe()` reveals central tendency and spread, and — crucially here — the
**minimum** of each column. A minimum of `0` for Glucose, BloodPressure, SkinThickness, Insulin,
or BMI is biologically impossible and flags the hidden missingness.

In [ ]:
summary = df.describe().T
summary["impossible_zero"] = summary.index.isin(ZERO_AS_MISSING)
summary.round(2)

**Takeaway:** the `min` for the five flagged columns is `0`, confirming that zeros are standing in
for missing measurements rather than being real values.

## 3. Missing-value analysis

*Why it matters:* quantifying how much is missing per column tells us whether imputation is
viable. Insulin and SkinThickness are notoriously incomplete in this dataset.

In [ ]:
zero_counts = (df[ZERO_AS_MISSING] == 0).sum()
missing = pd.DataFrame({
    "zeros": zero_counts,
    "pct_missing": (zero_counts / len(df) * 100).round(1),
}).sort_values("pct_missing", ascending=False)
display(missing)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=missing.index, y=missing["pct_missing"], ax=ax, color="#d62728")
ax.set_title("Missing values (encoded as 0) by column")
ax.set_ylabel("% missing")
ax.set_xlabel("")
for i, v in enumerate(missing["pct_missing"]):
    ax.text(i, v + 0.5, f"{v}%", ha="center")
plt.tight_layout()
plt.show()

**Takeaway:** Insulin (~49%) and SkinThickness (~30%) are heavily missing; Glucose, BMI, and
BloodPressure have only a handful. This range of missingness is exactly why the pipeline imputes
with the **median** (robust to the skew we see below) rather than dropping rows.

## 4. Target class balance

*Why it matters:* class imbalance affects model evaluation and whether we need stratified
splitting or resampling downstream.

In [ ]:
counts = df["Outcome"].value_counts().sort_index()
props = (counts / len(df) * 100).round(1)
print(counts.rename({0: "No diabetes (0)", 1: "Diabetes (1)"}))
print("\nProportions (%):")
print(props.rename({0: "No diabetes (0)", 1: "Diabetes (1)"}))

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(x="Outcome", data=df, ax=ax)
ax.set_xticklabels(["No diabetes (0)", "Diabetes (1)"])
ax.set_title("Target class balance")
plt.tight_layout()
plt.show()

**Takeaway:** roughly **65% / 35%** (500 vs 268). Mild imbalance — accuracy alone is misleading,
so downstream models should use stratified splits and metrics like F1/AUC.

## 5. Univariate distributions

*Why it matters:* the shape of each feature (skew, multimodality) guides imputation choice and
whether transformations or binning will help. Here we mask zeros so the distributions reflect
real measurements.

In [ ]:
df_nan = df.copy()
df_nan[ZERO_AS_MISSING] = df_nan[ZERO_AS_MISSING].replace(0, np.nan)
features = [c for c in df.columns if c != "Outcome"]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), features):
    sns.histplot(df_nan[col].dropna(), kde=True, ax=ax, color="#1f77b4")
    ax.set_title(col)
    ax.set_xlabel("")
fig.suptitle("Univariate distributions (zeros masked)", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

**Takeaway:** Insulin, DiabetesPedigreeFunction, and Age are right-skewed; Glucose, BMI, and
BloodPressure are roughly bell-shaped. Skew is why the **median** is the safer imputation choice.

## 6. Outlier detection

*Why it matters:* extreme values can distort models and may be data errors. We visualise with
boxplots and quantify with the IQR rule (points beyond 1.5×IQR from the quartiles).

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), features):
    sns.boxplot(y=df_nan[col], ax=ax, color="#2ca02c")
    ax.set_title(col)
    ax.set_ylabel("")
fig.suptitle("Boxplots (zeros masked)", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

def iqr_outliers(s):
    s = s.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((s < lo) | (s > hi)).sum()

outliers = pd.Series({c: iqr_outliers(df_nan[c]) for c in features}, name="iqr_outliers")
outliers.sort_values(ascending=False).to_frame()

**Takeaway:** Insulin, SkinThickness, and DiabetesPedigreeFunction carry the most extreme values.
These are plausible clinical extremes rather than obvious errors, so we keep them but note the
long tails for downstream scaling.

## 7. Correlation analysis

*Why it matters:* correlations reveal redundancy among features and which features relate most
to the target — useful for feature selection and for understanding multicollinearity.

In [ ]:
corr = df_nan.corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, ax=ax)
ax.set_title("Pearson correlation matrix")
plt.tight_layout()
plt.show()

print("Correlation with Outcome (sorted):")
corr["Outcome"].drop("Outcome").sort_values(ascending=False).round(3)

**Takeaway:** **Glucose** is by far the strongest correlate of `Outcome`, followed by BMI and
Age. No pair of features is so correlated as to be redundant, so all features are worth keeping.

## 8. Feature vs. target

*Why it matters:* comparing each feature's distribution across the two outcome classes shows
which features actually separate diabetic from non-diabetic patients.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), features):
    sns.boxplot(x="Outcome", y=col, data=df_nan, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
fig.suptitle("Feature distributions by Outcome (0 = no diabetes, 1 = diabetes)", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

**Takeaway:** Glucose, BMI, Age, and Pregnancies shift clearly upward for diabetic patients —
consistent with the correlation ranking and with clinical intuition.

## 9. Pairwise relationships

*Why it matters:* a pairplot of the most predictive features shows joint structure and how well
the classes separate in 2-D — a quick sense of how learnable the problem is.

In [ ]:
top_features = ["Glucose", "BMI", "Age", "Outcome"]
grid = sns.pairplot(df_nan[top_features].dropna(), hue="Outcome", diag_kind="kde", height=2.2)
grid.fig.suptitle("Pairwise relationships among top features", y=1.02)
plt.show()

**Takeaway:** higher Glucose combined with higher BMI/Age concentrates the diabetic class —
the classes overlap but are partially separable, so a non-trivial classifier should do well.

## 10. Key insights & pipeline justification

**What the EDA showed**

- **768 records, 8 features, binary target.** Mild class imbalance (~65% / 35%) → use stratified
  splits and threshold-aware metrics downstream.
- **Hidden missingness.** Zeros in Glucose / BloodPressure / SkinThickness / Insulin / BMI are
  impossible values. Insulin (~49%) and SkinThickness (~30%) are heavily affected.
- **Skewed, long-tailed features.** Insulin, DiabetesPedigreeFunction, and Age are right-skewed
  with notable outliers.
- **Strongest signals:** Glucose, then BMI and Age, correlate most with `Outcome`.

**How this justifies the `data_versioning` pipeline**

- `clean.py` replaces the impossible zeros with the column **median** — the right choice given the
  skew and missingness levels we measured (mean would be dragged by the long tails; dropping rows
  would discard ~half the data because of Insulin).
- `features.py` bins **BMI** into clinical categories and **Age** into decades — sensible because
  both show clear, monotonic separation by `Outcome`, and the `Glucose × BMI` interaction combines
  the two strongest predictors identified here.

EDA thus confirms the pipeline's transformations are grounded in the data, not arbitrary.